### Collections Model to Identify Potential Defaulters

In [1]:
#Import Libraries
import seaborn as sns
import pandas as pd
import numpy as np
import glob
import os
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [2]:
#Define the Model Periods
Month_Year = 'Aug_2023'

#Model Collection Files
Files = glob.glob(fr"D:\Strategy and Analytics\Collections\Collections Model\Model_Imports\Collections Reports\Collections_Serverfiles\{Month_Year}\Service_Points_Attached\*.csv")

#GOU Accounts
GoU = pd.read_csv(r"D:\Strategy and Analytics\Collections\Collections Model\Support Files\GOU Accounts\GOU_Accounts.csv",dtype='unicode')

#Active_DB
Colsn = ['SP_POINT_NO', 'REGION', 'DISTRICT', 'CONTRACT_NO','TARIFF', 'TARIFF_CLASS', 'CONTRACT_STATUS']
Active_Db = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Model\Support Files\Active_Db\*.csv")
db = pd.concat([pd.read_csv(File, dtype='unicode', encoding='latin-1') for File in Active_Db], ignore_index=True)
db.columns = ['REGION', 'DISTRICT', 'CONTRACT_NO', 'TARIFF_CODE', 'TARIFF', 'TARIFF_CLASS', 'CONTRACT_STATUS','SP_POINT_NO', 'CONNECTION_DATE','TERMINATION_DATE']
db =  db.apply(lambda x: x.str.strip() if isinstance(x,str) else x)
db = db[Colsn]

#Import Database With Service Points
Svs = pd.read_csv(r"D:\Strategy and Analytics\Collections\Collections Reports\All_Service_Points_DataBase.csv", dtype='unicode', encoding='latin-1')
Svs.columns=['SP_POINT_NO','CONTRACT_NO','Month_Year']

#Aged Debt Files
#Dont Delete - \\serverfile\Data-Extracts\DISTRICTS\DAILY\AGED_DEBTORS_LIST\2023\08
Cols = ['SP_POINT_NO','CONTRACT_NO','ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']
Debt_File = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Model\Support Files\Aged Debt\*.csv")
Debt = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Debt_File], ignore_index=True)

#Customer Names
Names = pd.read_csv(fr"D:\Strategy and Analytics\Collections\Collections Model\Support Files\Customer Names\PostPaid_Customer_Names_{Month_Year}.csv", dtype='unicode')

In [3]:
#Pick the Updated Debt File
try:
    Debt_File = glob.glob(r"\\serverfile\Data-Extracts\DISTRICTS\DAILY\AGED_DEBTORS_LIST\2023\08\*.csv")
    if 'consolidated' in File:
        Debt = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Debt_File], ignore_index=True)       
except:
    Debt_File = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Model\Support Files\Aged Debt\*.csv")
    Debt = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Debt_File], ignore_index=True)

In [4]:
#Collection Reports File Processing.
dfs = []
for File in Files:
    df = pd.read_csv(File, encoding='latin-1', dtype='unicode')
    File_name = os.path.basename(File)
    Filename = os.path.splitext(File_name)[0]
    df = df.dropna(subset=['REGION'], axis=0)
    df[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']] = df[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']].astype(float)
    df['Rate'] = df.apply(lambda row: 1 if row['BILLED_AMOUNT'] == 0 else (row['COLLECTED_AMOUNT'] / row['BILLED_AMOUNT']) if row['COLLECTED_AMOUNT'] != 0 else 0, axis=1)
    df['Rate'] = pd.to_numeric(df['Rate'], errors='coerce')
    df = df.drop_duplicates(subset='SP_POINT_NO')
    df = df[df['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
    df.fillna(1, inplace=True)
    df['Rate'] = df['Rate'].astype(float)
    dfs.append(df)
#Assign File Names to Dataframes
[df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023] = dfs

In [5]:
# Create a list of DataFrame names
df_names = ['df_2016', 'df_2017', 'df_2018', 'df_2019', 'df_2020', 'df_2021', 'df_2022', 'df_2023']
# Create lists to store data
data = []
# Iterate through the DataFrames and their names
for df, df_name in zip(dfs, df_names):
    Billed_Sum = df['BILLED_AMOUNT'].sum(axis=0)
    Collected_Sum = df['COLLECTED_AMOUNT'].sum(axis=0)
    data.append([df_name, Billed_Sum, Collected_Sum])
# Create a new DataFrame
result_df = pd.DataFrame(data, columns=['Month', 'Billed_Sum', 'Collected_Sum'])
# Print the resulting DataFrame
result_df['Rate'] = (result_df['Collected_Sum']/result_df['Billed_Sum'])*100
result_df

,Month,Billed_Sum,Collected_Sum,Rate
0,df_2016,102128421529.00,109619764249.00,107.34
1,df_2017,111250697206.94,112753924199.40,101.35
2,df_2018,121105941028.40,120815472610.40,99.76
3,df_2019,120666502691.80,135321231581.19,112.14
4,df_2020,110508747426.79,112242012152.07,101.57
5,df_2021,106096450238.49,107285890419.47,101.12
6,df_2022,119274378324.56,124695949357.46,104.55
7,df_2023,139222611975.64,25220727721.39,18.12


### Workings

In [6]:
#Workings
Ws = db.copy()
Ws = Ws[Ws['TARIFF_CLASS']!='OTHERS']
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')

#Compute and Attach Rates
Years = [2016, 2017, 2018, 2019, 2020, 2021, 2022]
Cols_2 = ['SP_POINT_NO','Rate']
for Year, df in zip(Years, dfs):
    Ws = pd.merge(Ws, dfs[Year-2016][Cols_2], on ='SP_POINT_NO', how='left')
    Ws = Ws.drop_duplicates(subset='CONTRACT_NO')
    Ws['Rate'].fillna(1, inplace=True)
    Ws['Rate'] = Ws['Rate'].astype(float)
    Ws.rename(columns={'Rate':f'Rate_{Year}'}, inplace=True)
    Ws.loc[:,'Rate_2016':] = Ws.loc[:,'Rate_2016':].fillna(1)
#Convert to Percentages
Ws.loc[:,'Rate_2016':] = Ws.loc[:,'Rate_2016':]*100

#Default Mappings
for Year in Years:
    Ws[f'Default_{Year}'] = Ws.apply(lambda row: 0 if row[f'Rate_{Year}'] == 0 else 2 if row[f'Rate_{Year}'] < 99.9 else 1, axis=1)

#Counts -  Defaults
Ws['Count'] = Ws.loc[:,'Default_2016':'Default_2022'].count(axis=1)
Ws['Counts_2'] = Ws.loc[:,'Default_2016':'Default_2022'].apply(lambda row: row.tolist().count(2), axis=1)
Ws['Counts_0'] =  Ws.loc[:,'Default_2016':'Default_2022'].apply(lambda row: row.tolist().count(0), axis=1)

#Probabilities
Ws['Prob_2'] = Ws.apply(lambda row: row['Counts_2']/row['Count'], axis=1)
Ws['Prob_0'] = Ws.apply(lambda row: row['Counts_0']/row['Count'], axis=1)

#Flat probability
Ws['Flat_Prob'] = Ws[['Prob_2','Prob_0']].sum(axis=1)

#Average Partial Probability
def Filter_and_Average(row):
    Filtered_Values = [x for x in row if 0 < x < 99.9]
    if len(Filtered_Values) > 0:
        average = sum(Filtered_Values) / len(Filtered_Values)
    else:
        average = 0       
    return average
# Apply the function to each row of the DataFrame
Ws['Avg_Partial'] = Ws.loc[:,'Rate_2018':'Rate_2022'].apply(Filter_and_Average , axis=1)

#Adjusted Probabilities
Ws['Weighted_Prob_2'] = ((100-Ws['Avg_Partial'])*Ws['Prob_2'])/100
Ws['Adj_Prob'] = Ws[['Prob_0','Weighted_Prob_2']].sum(axis=1)

#Default_Probability
Ws['Default_Probability'] = Ws['Adj_Prob'].apply(lambda x: 'Very High Probability of default' if x >= 0.5 else 'High Probability of default' if (0.3 <= x < 0.5) else 'Low Probability of default')

#Process Customer Names
Nms =  pd.concat([df_2023[['SP_POINT_NO','CUSTOMER_NAME']], Names[['SP_POINT_NO','CUSTOMER_NAME']]], axis=0, ignore_index=True)
Nms[['SP_POINT_NO','CUSTOMER_NAME']]  = Nms[['SP_POINT_NO','CUSTOMER_NAME']].astype(str)
Nms = Nms.drop_duplicates()

#Merge Customer Names
Ws =  pd.merge(Ws, Nms[['SP_POINT_NO','CUSTOMER_NAME']], on='SP_POINT_NO', how='left')
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')
Ws['Month_Year'] = Month_Year

#Attach Collected and Billed Amounts from Current Collections Report
Ws = pd.merge(Ws, df_2023[['SP_POINT_NO', 'SALE_CYCLE', 'BILLED_AMOUNT', 'COLLECTED_AMOUNT']], how='left', on='SP_POINT_NO')
Ws.fillna(0, inplace=True)
Ws = Ws.drop_duplicates(subset='CONTRACT_NO')

#Merge GOU Customers
Wsn = pd.merge(Ws,GoU[['CONTRACT_NO','Is_GOU?']], on='CONTRACT_NO', how='left')
Wsn['Is_GOU?'].fillna('NO',inplace=True)
Wsn = Wsn.drop_duplicates(subset='CONTRACT_NO')

In [7]:
#Attach service Points to Credit Risk
#Import the Credit Risk Files
Crs = []
Crn_s = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Model\Support Files\Credit_Risk\*.csv")
Cr = pd.concat([pd.read_csv(cr, dtype='unicode', encoding='latin-1') for cr in Crn_s], ignore_index=True)
Cr = Cr.iloc[:,[0,-1]]
Cr.columns = ['CONTRACT_NO', 'CREDIT_RISK']
Cr = Cr.drop_duplicates(subset='CONTRACT_NO')
Cr = Cr.reset_index(drop=True)
Credit_Risk =  pd.merge(Cr, Svs[['SP_POINT_NO','CONTRACT_NO']], on='CONTRACT_NO',how='left')
Credit_Risk = Credit_Risk[Credit_Risk['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Credit_Risk =  Credit_Risk.drop_duplicates(subset='CONTRACT_NO')

#Merge Workings and Credit Risk
Wsm =  pd.merge(Wsn, Credit_Risk[['SP_POINT_NO','CREDIT_RISK']], on='SP_POINT_NO', how='left')
Wsm['CREDIT_RISK'].fillna('0',inplace=True)
Wsm = Wsm.drop_duplicates(subset='CONTRACT_NO')

#Assign Credit Risk
def Assn_Credit_Risk(row):
    if row['CREDIT_RISK'] != '0':
        return row['CREDIT_RISK']
    elif (row['Default_Probability'] == 'Very High Probability of default') & (row['CREDIT_RISK'] == '0'): 
        return 'High Risk'
    elif (row['Default_Probability'] == 'High Probability of default') & (row['CREDIT_RISK'] == '0'):
        return 'Medium High Risk'
    else:
        'Low_Risk'
Wsm['CREDIT_RISK'] = Wsm.apply(Assn_Credit_Risk, axis=1)

#Merge Aged Debt Files and Workings
Debt[['ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']] = Debt[['ZERO_TO_THIRTY_DAYS', 'TOTAL_DEBT']].astype(float)
Debt['Arrears_Abv_30days'] = Debt['TOTAL_DEBT'] - Debt['ZERO_TO_THIRTY_DAYS']

#Add Aged Debt to Workings
Wsm = pd.merge(Wsm, Debt[['CONTRACT_NO','Arrears_Abv_30days']], on='CONTRACT_NO', how='left')
Wsm['Arrears_Abv_30days'].fillna(0,inplace=True)
Wsm = Wsm[Wsm['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Wsm = Wsm.drop_duplicates(subset='CONTRACT_NO')
Wsm['CUSTOMER_NAME'].fillna("0",inplace=True)

In [8]:
#Filters for Potential Defaulters
Filter1 = Wsm['Adj_Prob']>=0.3
Filter2 = Wsm['Is_GOU?']=='NO'
Filter3 = Wsm['TARIFF_CLASS']!='OTHERS'
Filter4 = Wsm['CREDIT_RISK'].isin(['Medium High Risk','High Risk'])

#Select Columns for Potential Defaulters
Colsn = ['SP_POINT_NO', 'REGION', 'DISTRICT', 'CONTRACT_NO', 'CUSTOMER_NAME','TARIFF', 'TARIFF_CLASS', 'CONTRACT_STATUS', 
 'SALE_CYCLE','Month_Year','Is_GOU?','BILLED_AMOUNT', 'COLLECTED_AMOUNT','CREDIT_RISK', 'Arrears_Abv_30days','Adj_Prob','Default_Probability']
#Filter Out the Potential defaulters
Pd = Wsm[Filter1&Filter2&Filter3&Filter4][Colsn]
Pd.reset_index(drop=True,inplace=True)
Pd = Pd[Pd['SP_POINT_NO'].apply(lambda x: len(str(x)) == 7)]
Pd = Pd.drop_duplicates(subset='SP_POINT_NO')
Pd.head(5)

,SP_POINT_NO,REGION,DISTRICT,CONTRACT_NO,CUSTOMER_NAME,TARIFF,TARIFF_CLASS,CONTRACT_STATUS,SALE_CYCLE,Month_Year,Is_GOU?,BILLED_AMOUNT,COLLECTED_AMOUNT,CREDIT_RISK,Arrears_Abv_30days,Adj_Prob,Default_Probability
0,2586690,KAMPALA_EAST,CSC_WANDEGEYA,200074860,SEWAKIRYANGA ANGEL,COMMERCIAL,TOU,ACTIVE,8,Aug_2023,NO,45447.94,0.00,Medium High Risk,17294.65,0.43,High Probability of default
1,2604946,KAMPALA_EAST,CSC_KITINTALE,200093116,ELECTORAL COMMISSION,INDUSTRIAL - LV,KVA,ACTIVE,8,Aug_2023,NO,17949742.83,0.00,Medium High Risk,21277906.22,0.45,High Probability of default
2,2611358,KAMPALA_WEST,CSC_NAJJANANKUMBI,200099528,LUYINDA MARIA MITI,COMMERCIAL,TOU,ACTIVE,8,Aug_2023,NO,12622.22,0.00,High Risk,434403.31,0.57,Very High Probability of default
3,2629850,KAMPALA_WEST,CSC_NATETE,200118020,KAGERE MOSES,COMMERCIAL,TOU,ACTIVE,8,Aug_2023,NO,4096.96,0.00,High Risk,7842662.59,0.38,High Probability of default
4,2658644,NORTH_EAST,CSC_TORORO,200146814,TORORO HOSPITAL .,INDUSTRIAL - LV,KVA,ACTIVE,8,Aug_2023,NO,9985227.01,0.00,Medium High Risk,35071451.25,0.71,Very High Probability of default


In [9]:
Pd.to_csv(fr"D:\Strategy and Analytics\Collections\Collections Model\Model Exports\Potential_Defaulters_{Month_Year}.csv", index=False)

In [10]:
len(Pd)

1064